In [1]:
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast
from huggingface_hub import hf_hub_download
import json
import pandas as pd
import stanza


c:\Users\marti\Diplomovka\SK_BPE_BLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Mapovanie UPOS na XPOS
upos_to_xpos = {
    "NOUN": "S",
    "PUNCT": "Z",
    "VERB": "V",
    "ADJ": "A",
    "ADP": "E",
    "PRON": "P",
    "PROPN": "S",
    "ADV": "D",
    "DET": "P",
    "AUX": "V",
    "CCONJ": "O",
    "PART": "T",
    "SCONJ": "O",
    "NUM": "N",
    "INTJ": "J",
    "SYM": "X",
    "X": "X"
}

In [7]:
class TokenClassifier:
    def __init__(self, model_name, tokenizer_name):
        # Načítanie modelu pre tokenovú klasifikáciu
        self.model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=14)
        # Inštancia tokenizeru s add_prefix_space=True pre prácu s predtokenizovanými vstupmi
        self.tokenizer = RobertaTokenizerFast.from_pretrained(tokenizer_name, add_prefix_space=True)
        # Načítanie bajtového mapovania (ak je potrebné pre dekódovanie)
        byte_utf8_mapping_path = hf_hub_download(repo_id=tokenizer_name, filename="byte_utf8_mapping.json")
        with open(byte_utf8_mapping_path, "r", encoding="utf-8") as f:
            self.byte_utf8_mapping = json.load(f)

    def classify_word(self, word):
        """
        Pre dané slovo (získané zo Stanza) vykoná zakódovanie s parametrom is_split_into_words=True.
        Ak sa slovo rozdelí na viacero sub-tokenov, ich logity sa priemerujú a určí sa výsledná značka.
        """
        # Zakódujeme slovo ako zoznam obsahujúci jedno slovo
        encoded = self.tokenizer([word.lower()], 
                                 is_split_into_words=True, 
                                 add_special_tokens=True, 
                                 return_tensors="pt")
        with torch.no_grad():
            output = self.model(**encoded)
        logits = output.logits  # tvar: (1, sekvencia, num_labels)

        # Získame word_ids pre vstup (mapuje tokeny na pôvodné slová)
        word_ids = encoded.word_ids(0)
        # Vyberieme tokeny, ktoré patria k nášmu slovu (ignorujeme None, ktoré patria špeciálnym tokenom)
        token_indices = [i for i, wid in enumerate(word_ids) if wid is not None]

        if not token_indices:
            # Ak nič nenájdeme, použijeme logity od prvého tokenu (malo by sa to stať zriedkavo)
            token_logits = logits[0][1:-1]
        else:
            token_logits = logits[0][token_indices]

        # Priemerujeme logity, ak je viac tokenov
        avg_logits = token_logits.mean(dim=0)
        pred_id = avg_logits.argmax().item()
        upos_tag = self.model.config.id2label[pred_id]
        xpos_tag = upos_to_xpos.get(upos_tag, "X")
        return word, xpos_tag

    def classify_words(self, words):
        """
        Prejde zoznam slov a pre každé slovo vráti jeho POS značku.
        """
        results = []
        for word in words:
            results.append(self.classify_word(word))
        return results

In [8]:
def main():
    # Načítanie textu zo súboru
    with open(r"C:\Users\marti\Diplomovka\DiploDiktaty.txt", 'r', encoding='utf-8') as file:
        text = file.read()

    # Inicializácia Stanza pipeline pre slovenský jazyk (iba na tokenizáciu)
    stanza.download('sk')
    nlp = stanza.Pipeline('sk', processors='tokenize')

    # Rozdelenie textu na vety a slová pomocou Stanza
    doc = nlp(text)
    sentences = [[word.text for word in sentence.words] for sentence in doc.sentences]

    # Inicializácia POS klasifikátora
    classifier = TokenClassifier(model_name="daviddrzik/SK_BPE_BLM-pos", tokenizer_name="daviddrzik/SK_BPE_BLM")

    # Spracovanie viet a zhromaždenie výsledkov
    data = []
    for sentence_id, words in enumerate(sentences, start=1):
        word_tags = classifier.classify_words(words)
        for token, xpos_tag in word_tags:
            data.append({
                'Token': token,
                'Xpos1': xpos_tag,
                'sentence_id': sentence_id
            })

    # Vytvorenie DataFrame
    df = pd.DataFrame(data)
    # Uloženie DataFrame do Excel súboru
    df.to_excel(r"C:\Users\marti\Diplomovka\outputSK_BPE_BLM-pos.xlsx", index=False, engine='openpyxl')

if __name__ == "__main__":
    main()

2025-02-18 17:38:51 INFO: Downloaded file to C:\Users\marti\stanza_resources\resources.json
2025-02-18 17:38:51 INFO: Downloading default packages for language: sk (Slovak) ...
2025-02-18 17:38:51 INFO: File exists: C:\Users\marti\stanza_resources\sk\default.zip
2025-02-18 17:38:52 INFO: Finished downloading models and saved to C:\Users\marti\stanza_resources
2025-02-18 17:38:52 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-02-18 17:38:52 INFO: Downloaded file to C:\Users\marti\stanza_resources\resources.json
2025-02-18 17:38:52 WARNING: Language sk package default expects mwt, which has been added
2025-02-18 17:38:52 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-02-18 17:38:52 INFO: Using device: cpu
2025-02-18 17:38:52 INFO: 